
# Task 2: Time-Series Regression (Remaining Useful Life/RUL)

## 1. Project Overview

### 1.1 Problem Definition

This project focuses on predicting the **Remaining Useful Life (RUL)** of press machines using time‑series sensor measurements collected throughout operational cycles. 

The primary objective is to estimate the number of remaining operating cycles before machine failure occurs, enabling valuable information for maintenance scheduling and reliability management. 

To improve model practicality and stability, the target variable is capped at a **maximum value of one hundred fifty operating cycles**.

$
RUL_t = \min(T_{\text{failure}} - t, 150)$

### 1.2 Project Objectives

1. Perform Exploratory Data Analysis (EDA).
2. Conduct data preprocessing and RUL label generation.
3. Transform time-series data into sliding windows.
4. Develop and compare sequence-based Deep Learning models.
5. Establish a non-Deep Learning baseline.
6. Perform machine-level validation.
7. Estimate prediction uncertainty.
8. Evaluate models using RMSE and Asymmetric Score.

### 1.3 Evaluation Metrics

- Root Mean Squared Error (RMSE)
- Asymmetric Score

### 1.4 Project Requirements

- At least two sequence-based Deep Learning architectures.
- One non-Deep Learning baseline.
- Machine-level train-validation split.
- Prediction intervals using Monte Carlo Dropout or Ensemble.
- Final RUL prediction for test machines.



## 2. Library Imports

Several Python libraries are imported to facilitate data processing, file operations, archive handling, and formatted output generation, forming the foundation for subsequent project implementations:

- **Pandas:** Used for data manipulation and tabular data analysis.
- **Pathlib:** Used to manage file and directory paths.
- **IPython.display:** Provides functions to display formatted outputs and render Markdown content within the notebook.
- **ZipFile:** Used to access and read files directly from a ZIP archive without manually extracting its contents.

Additional libraries will be imported in subsequent sections depending on the requirements of data analysis, preprocessing, visualization, and model development.


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display, Markdown
from zipfile import ZipFile

## 3. Dataset Understanding

### 3.1 Dataset Archive Inventory

The following code inspects the compressed dataset archive and builds a formatted summary table displaying each file name along with its corresponding storage size in megabytes.

In [24]:
zip_path = Path(r"C:\Users\LENOVO\OneDrive\Documents\Telkom University\SEMESTER 9\Deep Learning\UTS\data_uts_dl.zip")

if zip_path.exists():
    with ZipFile(zip_path, "r") as zip_ref: #r means read, zip_ref is for the zip folder that opened
        files = [item for item in zip_ref.infolist() if not item.is_dir()] #infolist takes zip lists, dir represent if the item a file not a dictionary
        inventory = pd.DataFrame([{
        "No": idx + 1,
        "File Name": file.filename,
        "File Size (MB)": file.file_size / (1024 * 1024) #byte to mb
        } for idx, file in enumerate(files)])

    display(
        inventory.style
        .set_caption("Dataset Inventory")
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("text-align", "center"),
                    ("font-size", "14px"),
                    ("font-weight", "bold"),
                    ("caption-side", "top"),
                    ("padding-bottom", "8px")
                ]},
            {"selector": "th", "props": [("text-align", "center")]}
        ])
        .format({"File Size (MB)": "{:.2f}"})
        .set_properties(**{"text-align": "center"})
        .set_properties(subset=["File Name"], **{"text-align": "left"})
        .hide(axis="index")
    )

else: 
    print("Archive data_uts_dl.zip not found at local path. Proceeding with data/ folder.")

No,File Name,File Size (MB)
1,klasifikasi_train_meta.csv,1.73
2,klasifikasi_test.npz,58.27
3,kamus_data.md,0.00
4,klasifikasi_train.npz,170.99
5,rul_test.csv.gz,3.94
6,rul_train.csv.gz,19.40
7,clustering_meta.csv,0.57
8,klasifikasi_test_meta.csv,0.46
9,clustering.npz,124.22


### 3.2 Data Dictionary Inspection

The following code extracts and renders the official data dictionary from the archive so that all variable definitions and target labeling rules remain clearly documented in the notebook.

In [26]:
kamusData = (Path.cwd().parent if Path.cwd().name.lower() == "kode" else Path.cwd()) / "kamus_data.md"

if zip_path.exists():
    with ZipFile(zip_path, "r") as zip_ref:
        kamus_data = next(
            name for name in zip_ref.namelist()
            if name.lower().endswith("kamus_data.md")
        )
        kamus_text = zip_ref.read(kamus_data).decode("utf-8-sig", errors="replace")
    display(Markdown(kamus_text))
elif kamusData.exists():
    display(Markdown(kamusData.read_text(encoding="utf-8")))
else:
    print("kamus_data.md not found locally.")

# Kamus data UTS Deep Learning

Semua kelompok menerima data yang sama. Label data uji tidak dibagikan.

## Tugas 1: klasifikasi_train.npz, klasifikasi_test.npz, klasifikasi_*_meta.csv

Satu sampel per unit produk dari pabrik komponen fiktif. Baris ke-i di berkas meta
memakai `indeks` = i pada array di berkas npz.

| Isi | Keterangan |
| --- | --- |
| `gambar` | Array uint8 (N, 64, 64), foto permukaan produk |
| `sensor` | Array float16 (N, 3, 128): suhu (C), getaran (relatif), arus (A) selama penekanan. NaN berarti data tidak terekam |
| id_unit | ID unit produk |
| lini | Lini produksi (A atau B) |
| shift | Shift produksi |
| id_mesin | Mesin penekan |
| batch_material | Batch bahan baku |
| kelas | Label: normal, gores, noda, retak, gelembung, las_dingin, kontaminasi. Hanya di data latih |

Label data latih berasal dari inspektur dan tidak dijamin bebas salah.

## Tugas 2: rul_train.csv.gz dan rul_test.csv.gz

Log per siklus untuk mesin press fiktif. Data latih berisi riwayat sampai mesin rusak.
Data uji berhenti pada titik acak sebelum rusak.

| Kolom | Keterangan |
| --- | --- |
| id_unit | ID mesin |
| siklus | Nomor siklus sejak mesin mulai dipakai |
| setel_1, setel_2, setel_3 | Pengaturan operasi pada siklus itu |
| s01-s21 | Bacaan sensor. NaN berarti bacaan hilang |

Label: sisa umur pakai (RUL) = jumlah siklus tersisa sampai rusak, dipotong pada 150
(sisa umur di atas 150 siklus dihitung 150). Di data latih, siklus terakhir tiap unit adalah
siklus saat rusak, jadi labelnya dihitung sendiri. Prediksi RUL dibuat pada siklus terakhir
tiap unit di data uji.

## Tugas 3: clustering.npz dan clustering_meta.csv

Sampel material dari laboratorium fiktif, tanpa label.

| Isi | Keterangan |
| --- | --- |
| `gambar` | Array uint8 (N, 64, 64), citra mikrostruktur dari mikroskop |
| `spektrum` | Array float16 (N, 128), spektrum komposisi |
| id_sampel | ID sampel |
| mikroskop | Mikroskop yang dipakai (M1 atau M2) |


## 4. Data Loading and Verification

### 4.1 Dataset Extraction and Loading

The following code verifies the repository directory structure, extracts the training and testing archives when necessary, and loads both datasets into structured DataFrame objects.

In [27]:
cwd = Path.cwd()
project_dir = cwd.parent if cwd.name.lower() == "kode" else cwd
data_dir = project_dir / "data"
data_dir.mkdir(parents=True, exist_ok=True)

train_path = data_dir / "rul_train.csv.gz"
test_path = data_dir / "rul_test.csv.gz"

if not (train_path.exists() and test_path.exists()):
    with ZipFile(zip_path, "r") as zip_ref:
        files = zip_ref.namelist()
        train_file = next(name for name in files if name.lower().endswith("rul_train.csv.gz"))
        test_file = next(name for name in files if name.lower().endswith("rul_test.csv.gz"))
        train_path.write_bytes(zip_ref.read(train_file))
        test_path.write_bytes(zip_ref.read(test_file))

print("Current working directory:", cwd)
print("Data directory           :", data_dir)
print("Train file exists        :", train_path.exists())
print("Test file exists         :", test_path.exists())

train = pd.read_csv(train_path, compression="gzip")
test = pd.read_csv(test_path, compression="gzip")

Current working directory: c:\Users\LENOVO\OneDrive\Documents\Telkom University\SEMESTER 9\Deep Learning\UTS\Deep-Learning-Midterm-Group-6\kode
Data directory           : c:\Users\LENOVO\OneDrive\Documents\Telkom University\SEMESTER 9\Deep Learning\UTS\Deep-Learning-Midterm-Group-6\data
Train file exists        : True
Test file exists         : True


### 4.2 Dataset Structure and Schema Verification

The following code inspects dataset dimensions, displays sample observations, and verifies that all required columns and unique machine identifiers match the official task specifications.

In [28]:
expected_columns = ["id_unit", "siklus", "setel_1", "setel_2", "setel_3"] + [f"s{i:02d}" for i in range(1, 22)]

missing_train = [col for col in expected_columns if col not in train.columns]
missing_test = [col for col in expected_columns if col not in test.columns]

print(f"Training shape           : {train.shape}")
print(f"Testing shape            : {test.shape}")
print(f"Unique training machines : {train['id_unit'].nunique()}")
print(f"Unique testing machines  : {test['id_unit'].nunique()}")
print(f"Missing train columns    : {missing_train}")
print(f"Missing test columns     : {missing_test}\n")

train.info()
display(train.head())
display(test.head())

Training shape           : (254174, 26)
Testing shape            : (51692, 26)
Unique training machines : 900
Unique testing machines  : 300
Missing train columns    : []
Missing test columns     : []

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 254174 entries, 0 to 254173
Data columns (total 26 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id_unit  254174 non-null  int64  
 1   siklus   254174 non-null  int64  
 2   setel_1  254174 non-null  float64
 3   setel_2  254174 non-null  float64
 4   setel_3  254174 non-null  float64
 5   s01      249093 non-null  float64
 6   s02      248902 non-null  float64
 7   s03      249058 non-null  float64
 8   s04      248959 non-null  float64
 9   s05      249065 non-null  float64
 10  s06      248939 non-null  float64
 11  s07      248955 non-null  float64
 12  s08      249044 non-null  float64
 13  s09      249001 non-null  float64
 14  s10      249036 non-null  float64
 15  s11      249137 non

,id_unit,siklus,setel_1,setel_2,setel_3,s01,s02,s03,s04,s05,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,3,1,20.001,0.7005,100.0,1107.746,569.289,991.700,1099.533,NaN,...,1092.248,1144.779,1345.634,873.087,851.099,408.142,997.673,838.331,1012.076,439.003
1,3,2,34.996,0.8424,100.0,1071.040,587.821,1013.673,NaN,157.684,...,1095.511,1167.320,1340.661,871.230,791.396,398.338,980.601,814.985,1108.741,412.368
2,3,3,0.001,-0.0012,100.0,1130.132,585.094,998.221,1127.436,161.918,...,1091.245,1170.340,1330.785,872.114,807.389,388.304,938.802,847.512,1003.616,455.443
3,3,4,35.001,0.8375,100.0,1080.962,585.775,1016.998,1152.147,NaN,...,1096.911,1173.207,1348.423,877.459,784.038,399.505,978.649,813.737,1113.184,411.554
4,3,5,9.998,0.2483,100.0,1095.013,580.506,997.108,1119.245,169.359,...,1102.956,1159.881,1341.195,872.265,772.284,NaN,950.568,821.425,969.681,362.551


,id_unit,siklus,setel_1,setel_2,setel_3,s01,s02,s03,s04,s05,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,20.001,0.7025,100.0,1109.194,566.947,992.053,1101.107,173.988,...,1092.054,1134.754,1347.409,872.737,846.643,409.136,995.276,836.568,1012.850,437.922
1,1,2,9.998,0.2521,100.0,1091.159,578.471,1002.437,1123.170,169.325,...,1103.964,1153.523,1342.812,870.304,775.113,381.319,948.034,821.784,966.125,362.720
2,1,3,42.002,0.8370,100.0,1086.341,557.209,990.383,1139.700,162.133,...,1103.025,1192.780,1315.747,877.673,802.890,373.825,1020.731,841.189,1019.764,402.863
3,1,4,0.002,0.0004,100.0,1129.872,585.203,1004.457,1126.587,161.752,...,1091.025,1167.104,1332.974,870.364,802.731,388.525,939.540,846.664,1003.844,454.214
4,1,5,24.998,0.6178,60.0,1115.094,608.829,1043.742,1129.542,186.207,...,1105.461,1217.908,1368.876,877.994,789.337,383.327,1016.059,883.760,987.725,462.625
